## Round 1 – System Design (Modern Data Stack)

1️⃣ Design a **Medallion Architecture (Bronze → Silver → Gold)**

2️⃣ **Batch vs Streaming** — when to use each in financial systems?

3️⃣ Build a pipeline for **real-time fraud detection (Kafka + Spark Streaming)**

4️⃣ How do you handle **schema drift & late-arriving data?**

5️⃣ Data governance: **data lineage, audit, and access control (RBAC)**



##1. Medallion Architecture: Bronze → Silver → Gold

A **Medallion Architecture** organizes data into multiple layers so that raw data gradually becomes **clean, trusted, and business-ready**.


```text
Sources
  │
  ├── APIs
  ├── Databases
  ├── Files
  └── Streaming
       │
       ▼
┌─────────────────────┐
│      BRONZE         │
│   Raw / Immutable   │
└─────────────────────┘
       │
       ▼
┌─────────────────────┐
│      SILVER         │
│ Cleaned / Validated │
└─────────────────────┘
       │
       ▼
┌─────────────────────┐
│       GOLD          │
│ Business / Analytics│
└─────────────────────┘
       │
       ▼
 BI / Reports / ML / Analytics
```

---

### 🥉 1. Bronze Layer — Raw Data

**Purpose:** Store data as close to the source as possible.

Examples:

* API responses
* CSV/JSON files
* Database CDC
* Streaming events

Typical operations:

* Minimal transformation
* Add ingestion timestamp
* Add source/file metadata
* Preserve the original data
* Handle basic schema evolution

Example:

```python
bronze_df = (
    spark.read
    .format("json")
    .load("/landing/transactions")
)

bronze_df = bronze_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

bronze_df.write.format("delta").mode("append").save(
    "/bronze/transactions"
)
```

**Key principle:** Don't aggressively transform Bronze data because it should provide a recoverable copy of the source.

---

### 🥈 2. Silver Layer — Cleaned & Validated

**Purpose:** Convert raw data into reliable, standardized datasets.

Typical transformations:

* Remove duplicates
* Handle NULLs
* Data type conversion
* Standardize values
* Apply business/data-quality rules
* Join related datasets
* Implement CDC/SCD logic

Example:

```python
silver_df = (
    bronze_df
    .dropDuplicates(["transaction_id"])
    .filter("amount >= 0")
    .withColumn("amount", col("amount").cast("decimal(18,2)"))
)
```

You might also maintain rejected records separately:

```text
Bronze
  │
  ├── Valid records ──→ Silver
  │
  └── Invalid records → Quarantine
```

---

### 🥇 3. Gold Layer — Business-Ready

**Purpose:** Create datasets optimized for analytics and business consumption.

Examples:

```text
Gold
 ├── Daily Sales
 ├── Customer 360
 ├── Product Performance
 ├── Revenue Dashboard
 └── Monthly KPI
```

Example:

```python
gold_df = (
    silver_df
    .groupBy("customer_id")
    .agg(
        sum("amount").alias("total_revenue"),
        count("transaction_id").alias("transaction_count")
    )
)
```

Gold tables are consumed by:

* Power BI
* Tableau
* SQL analytics
* Data science/ML
* Business applications

---

## Production Design

For a real-world data engineering project, I would design it like:

```text
                 ┌──────────────┐
                 │ API / Oracle │
                 │ DB / SFTP    │
                 └──────┬───────┘
                        │
                 ADF / Auto Loader
                        │
                        ▼
              ┌──────────────────┐
              │ Bronze Delta     │
              │ Raw + Metadata   │
              └────────┬─────────┘
                       │
                 PySpark / DLT
                       │
                       ▼
              ┌──────────────────┐
              │ Silver Delta     │
              │ Clean + Quality  │
              └────────┬─────────┘
                       │
                  Business Logic
                       │
                       ▼
              ┌──────────────────┐
              │ Gold Delta       │
              │ Aggregations/KPI │
              └────────┬─────────┘
                       │
                       ▼
                 Power BI / BI
```

### Important Design Considerations

**Incremental Processing:** Use CDC, watermarks, or ingestion timestamps instead of processing the entire dataset every time.

**Data Quality:** Validate schema, duplicates, NULLs, referential integrity, and business rules in Silver.

**Security:** Apply appropriate access controls at each layer and protect sensitive columns.

**Performance:** Use partition pruning, optimized file sizes, appropriate clustering/Z-Ordering, and efficient joins.

**ACID & Reliability:** Delta Lake provides transactional guarantees and supports reliable updates, deletes, and schema evolution.

### 🔥 Interview Answer

> **"I would design the Medallion Architecture with Bronze, Silver, and Gold layers. Bronze stores raw and immutable source data with ingestion metadata. Silver performs cleansing, deduplication, schema standardization, data-quality checks, CDC and business transformations. Gold contains business-ready aggregated datasets optimized for reporting and analytics. I would implement incremental processing, Delta Lake for ACID transactions, monitoring and data-quality checks, and appropriate partitioning and optimization at each layer."**


##2. Batch vs Streaming in Financial Systems

The choice depends mainly on **latency, transaction volume, business criticality, and regulatory requirements**.

| Aspect     | Batch Processing               | Streaming               |
| ---------- | ------------------------------ | ----------------------- |
| Processing | Periodic                       | Continuous              |
| Latency    | Minutes to hours               | Milliseconds to seconds |
| Cost       | Generally lower                | Generally higher        |
| Complexity | Lower                          | Higher                  |
| Best for   | Historical/periodic processing | Real-time decisions     |
| Example    | Daily reconciliation           | Fraud detection         |

### 🟦 Batch Processing

Data is collected and processed at scheduled intervals.

**Financial use cases:**

* Daily account statements
* End-of-day ledger processing
* Daily/monthly reconciliation
* Regulatory reporting
* Historical data loads
* Payroll/interest calculations

Example:

```text
Bank Transactions
       ↓
  Daily Batch
       ↓
Data Validation
       ↓
Reconciliation
       ↓
Financial Reports
```

If a report only needs to be generated once every night, streaming would add unnecessary complexity.

---

### 🟢 Streaming Processing

Transactions are processed continuously as events arrive.

**Financial use cases:**

* Real-time fraud detection
* Credit-card transaction monitoring
* Suspicious transaction detection
* Real-time balance updates
* Payment processing
* Real-time alerts
* Market/stock-price processing

Example:

```text
Transaction
     ↓
Kafka / Event Hub
     ↓
Spark Structured Streaming
     ↓
Fraud Rules / ML Model
     ↓
Alert / Block Transaction
```

For example, if a card transaction occurs in Bengaluru and another transaction occurs in New York two minutes later, a streaming system can evaluate the events immediately and potentially trigger a fraud alert.

---

### ⭐ Hybrid Approach — Common in Financial Systems

In practice, I would often use **both**.

```text
                  Transactions
                       │
             ┌─────────┴─────────┐
             ↓                   ↓
        Streaming              Batch
             ↓                   ↓
    Fraud Detection        EOD Processing
    Real-time Alerts       Reconciliation
    Balance Updates        Regulatory Reports
             │                   │
             └─────────┬─────────┘
                       ↓
                 Data Lakehouse
```

For example:

* **Streaming:** Detect fraud within seconds.
* **Batch:** Perform end-of-day reconciliation between transaction systems and the ledger.
* **Both:** Store the events in a lakehouse for historical analytics and auditing.

### 🔥 Interview Answer

> **"In financial systems, I use streaming when the business requires immediate action, such as fraud detection, payment processing, real-time alerts, or balance updates. I use batch processing for workloads where latency isn't critical, such as end-of-day reconciliation, financial reporting, regulatory reporting, and historical processing. For most large financial platforms, I prefer a hybrid architecture where streaming handles real-time events and batch handles reconciliation and periodic reporting. I would also design for exactly-once or effectively-once processing, idempotency, auditability, fault tolerance, and data recovery because financial transactions require strong correctness guarantees."**


##3. Real-Time Fraud Detection Pipeline — Kafka + Spark Structured Streaming

A production-grade design would look like this:

```text
                ┌──────────────────┐
                │ Card / Payment   │
                │ Transactions     │
                └────────┬─────────┘
                         │
                         ▼
                 ┌───────────────┐
                 │     Kafka     │
                 │ transaction   │
                 │    topic      │
                 └───────┬───────┘
                         │
                         ▼
              ┌─────────────────────┐
              │ Spark Structured    │
              │ Streaming           │
              └──────────┬──────────┘
                         │
              ┌──────────▼──────────┐
              │ Parse + Validate    │
              │ Schema / Data       │
              │ Quality             │
              └──────────┬──────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │ Enrichment          │
              │ Customer / Merchant │
              │ Reference Data      │
              └──────────┬──────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │ Fraud Rules / ML    │
              │ Feature Calculation │
              └──────────┬──────────┘
                         │
                  ┌──────┴───────┐
                  ▼              ▼
             FRAUD           LEGITIMATE
                │                 │
                ▼                 ▼
          Alert / Block       Transaction
          Transaction         Storage
```

### 1. Transaction Event

Suppose Kafka receives:

```json
{
  "transaction_id": "TX10001",
  "customer_id": "C101",
  "merchant_id": "M500",
  "amount": 95000,
  "timestamp": "2026-09-01T20:30:00",
  "location": "Bengaluru"
}
```

---

### 2. Read From Kafka

```python
from pyspark.sql import functions as F
from pyspark.sql.types import *

schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("merchant_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("timestamp", TimestampType()),
    StructField("location", StringType())
])

events = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "latest")
    .load()
)
```

Parse the Kafka value:

```python
transactions = (
    events
    .select(
        F.from_json(
            F.col("value").cast("string"),
            schema
        ).alias("data")
    )
    .select("data.*")
)
```

---

### 3. Apply Fraud Rules

For example, flag transactions above ₹1 lakh:

```python
fraud_transactions = transactions.filter(
    F.col("amount") > 100000
)
```

But real fraud detection should use **multiple signals**, such as:

```text
High transaction amount
        +
Multiple transactions in short period
        +
Unusual customer location
        +
New merchant/device
        +
Previous fraud history
        ↓
Fraud Score
```

---

### 4. Detect Multiple Transactions in a Time Window

For example, count transactions per customer over a 5-minute window:

```python
velocity = (
    transactions
    .withWatermark("timestamp", "10 minutes")
    .groupBy(
        F.window("timestamp", "5 minutes"),
        "customer_id"
    )
    .agg(
        F.count("*").alias("txn_count"),
        F.sum("amount").alias("total_amount")
    )
)
```

Then flag suspicious activity:

```python
suspicious = velocity.filter(
    (F.col("txn_count") >= 5) |
    (F.col("total_amount") > 200000)
)
```

---

### 5. Watermarking

Watermarks are important because financial events can arrive **late or out of order**.

```python
transactions = transactions.withWatermark(
    "timestamp",
    "10 minutes"
)
```

This allows Spark to handle late events while preventing unlimited state from accumulating.

---

### 6. Write Fraud Alerts

Fraud events can be written to a downstream Kafka topic:

```python
alerts = (
    suspicious
    .select(
        F.col("customer_id").cast("string").alias("key"),
        F.to_json(
            F.struct("*")
        ).alias("value")
    )
)

query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "fraud-alerts")
    .option("checkpointLocation", "/checkpoints/fraud")
    .start()
)
```

A downstream service could consume `fraud-alerts` and **block the transaction, notify the fraud team, or request additional authentication**.

---

## Production Considerations

### Exactly-once / Idempotency

Financial systems must avoid duplicate financial actions. I would use:

* Kafka offsets
* Spark checkpoints
* Unique `transaction_id`
* Idempotent downstream processing

The key distinction is that **Spark's processing guarantees alone don't automatically make every external side effect exactly-once**. The sink/action needs to be designed for idempotency as well.

### Fault Tolerance

Use:

```text
Kafka
  ↓
Spark Streaming
  ↓
Checkpoint
  ↓
Restart from last committed progress
```

If the Spark job fails, checkpointing allows it to recover its streaming progress.

### Performance

I would optimize using:

* Appropriate Kafka partitions
* Parallel Spark processing
* Efficient state management
* Broadcast joins for small reference data
* AQE where applicable
* Avoiding unnecessary shuffles
* Monitoring consumer lag and Spark Streaming metrics

---

## 🔥 Interview Answer

> **"I would ingest transaction events into Kafka and use Spark Structured Streaming to consume them continuously. Spark would validate and parse the events, enrich them with customer and merchant information, and calculate real-time fraud features such as transaction velocity, amount thresholds, location changes, and historical behavior. I would use watermarking for late-arriving events and maintain state for time-window calculations. Transactions exceeding the fraud rules or ML score threshold would be published to a fraud-alert topic for blocking or investigation, while legitimate transactions would be stored in the lakehouse. For reliability, I would use checkpoints, unique transaction IDs, idempotent processing, monitoring, and proper partitioning."**


##4. Schema Drift & Late-Arriving Data in PySpark

These are two common challenges in **batch and streaming pipelines**, especially in financial systems.

### 1. Schema Drift

**Schema drift** occurs when the source schema changes unexpectedly.

Examples:

```text
Before:
customer_id | amount | transaction_date

After:
customer_id | amount | transaction_date | currency
```

Or:

```text
amount → string
```

### How I Handle It

**Step 1 — Schema validation**

Compare the incoming schema with the expected schema.

```python
expected_columns = {
    "transaction_id",
    "customer_id",
    "amount",
    "transaction_date"
}

incoming_columns = set(df.columns)

missing = expected_columns - incoming_columns
new_columns = incoming_columns - expected_columns

print("Missing:", missing)
print("New:", new_columns)
```

**Step 2 — Handle new columns**

If the new column is acceptable, evolve the schema rather than failing the entire pipeline.

With Delta Lake:

```python
(
    df.write
      .format("delta")
      .mode("append")
      .option("mergeSchema", "true")
      .save("/silver/transactions")
)
```

I would **not blindly enable schema evolution** for every pipeline. Critical schema changes should go through validation/governance.

**Step 3 — Handle missing columns**

Add them with a default or NULL:

```python
from pyspark.sql.functions import lit

if "currency" not in df.columns:
    df = df.withColumn("currency", lit(None).cast("string"))
```

---

# 2. Late-Arriving Data

Late-arriving data means an event arrives **after its actual event time**.

Example:

```text
Event Time:    10:02 AM
Arrival Time:  10:15 AM
```

This is especially common in streaming systems because of network delays, retries, or upstream failures.

### Use Watermarking

In Structured Streaming:

```python
stream_df = (
    stream_df
    .withWatermark("event_time", "15 minutes")
)
```

Then perform a time-window aggregation:

```python
result = (
    stream_df
    .withWatermark("event_time", "15 minutes")
    .groupBy(
        window("event_time", "5 minutes"),
        "customer_id"
    )
    .agg(
        sum("amount").alias("total_amount")
    )
)
```

The **15-minute watermark** tells Spark that it should allow late events within that threshold while eventually cleaning up old state.

---

### Batch Late-Arriving Data

For batch pipelines, I would use:

* Incremental load based on event/update timestamp
* Reprocessing of affected partitions
* MERGE/UPSERT
* Idempotent processing

For example, with Delta:

```sql
MERGE INTO target t
USING source s
ON t.transaction_id = s.transaction_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;
```

This allows a late transaction to be inserted or an existing record to be corrected.

---

## 🔥 Interview Answer

> **"For schema drift, I first validate the incoming schema against the expected schema and classify changes as additions, deletions, or datatype changes. For approved additive changes, I can use controlled schema evolution, while breaking changes are quarantined or rejected. For late-arriving data, I use event time rather than processing time, watermarking in Structured Streaming, and appropriate time windows. In batch pipelines, I reprocess affected partitions or use MERGE/UPSERT based on business keys. I also make the pipeline idempotent so that retries and late data don't create duplicates."**

### Quick Memory Trick

**Schema Drift → Validate → Evolve → Quarantine breaking changes**

**Late Data → Event Time → Watermark → Reprocess/MERGE**


##5. Data Governance: Lineage, Audit & RBAC

In a production data platform, I would implement governance across **who can access data, where the data came from, and what happened to it**.

### 1. Data Lineage

**Data lineage** tracks the movement and transformation of data from source to consumption.

```text
Oracle / API / Kafka
        ↓
     Bronze
        ↓
     Silver
        ↓
      Gold
        ↓
   Power BI / ML
```

For example:

```text
Oracle.customer
      ↓
Bronze.customer_raw
      ↓
Silver.customer
      ↓
Gold.customer_360
      ↓
Power BI Dashboard
```

Lineage helps answer:

* Where did this data come from?
* Which transformations were applied?
* Which reports depend on this table?
* What will be impacted if a source column changes?

For a **Databricks/Lakehouse environment**, I would capture lineage through the platform's catalog/governance capabilities and maintain metadata for datasets, columns, jobs, and dependencies.

---

## 2. Audit

Auditing tracks **who accessed or changed data and when**.

A typical audit record could contain:

```text
user_id
timestamp
table_name
operation
job_name
status
rows_processed
source_system
run_id
```

Example:

```sql
SELECT
    user_id,
    table_name,
    operation,
    event_timestamp,
    status
FROM audit_log
WHERE table_name = 'customer_transactions'
ORDER BY event_timestamp DESC;
```

For ETL pipelines, I would also maintain an **ETL audit table**:

```text
pipeline_name
run_id
start_time
end_time
source_count
target_count
error_count
status
```

This makes reconciliation and troubleshooting much easier.

---

## 3. Access Control — RBAC

**RBAC = Role-Based Access Control.**

Instead of granting permissions individually to every user, I create roles based on responsibilities.

```text
                 Data Platform
                      │
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
    Data Engineer   Analyst      Business User
        │             │             │
    Bronze/Silver    Gold        Gold/Reports
      Read/Write      Read           Read
```

For example:

| Role          | Bronze     | Silver     | Gold |
| ------------- | ---------- | ---------- | ---- |
| Data Engineer | Read/Write | Read/Write | Read |
| Data Analyst  | Read       | Read       | Read |
| Business User | No access  | No access  | Read |

Sensitive data should receive additional controls such as:

* Column-level permissions
* Row-level security
* Data masking
* Encryption
* Secrets management

For example, an analyst may see:

```text
customer_id | name | email_masked | transaction_amount
```

instead of the raw sensitive fields.

---

## Production Governance Flow

```text
             Data Sources
                  ↓
          ┌───────────────┐
          │   Ingestion   │
          └───────┬───────┘
                  ↓
              Bronze
                  ↓
              Silver
                  ↓
               Gold
                  ↓
             BI / ML
                  │
     ┌────────────┼────────────┐
     ↓            ↓            ↓
  Lineage       Audit        RBAC
     ↓            ↓            ↓
Data movement  User/action  Permissions
```

### 🔥 Interview Answer

> **"For data governance, I focus on three areas: lineage, audit, and access control. Data lineage tracks data from source through Bronze, Silver, and Gold to the final consumption layer, helping with impact analysis and troubleshooting. For auditing, I maintain pipeline and access logs containing information such as user, timestamp, operation, run ID, record counts, and status. For access control, I implement RBAC so users receive permissions based on their roles rather than individual grants. For sensitive data, I additionally use column-level controls, masking, row-level security, encryption, and secrets management."**


## Round 2 – Hiring Manager

1️⃣ Deep dive into your **end-to-end pipeline design**

2️⃣ A time you handled **production failure or data incident**

3️⃣ Trade-offs: **performance vs cost vs reliability**

4️⃣ Working with stakeholders in **high-pressure environments**

5️⃣ How you design **scalable systems for TB-level data**

1. Deep dive into your end-to-end pipeline design

### 1. Source Systems

* Oracle / SQL Server / APIs / SFTP / Files
* Identify **full vs incremental/CDC** extraction strategy.
* Capture metadata such as watermark, load date, source file name.

### 2. ADF – Orchestration

* ADF acts as the **orchestration layer**.
* Use **Linked Services + Datasets + Parameters**.
* Trigger Databricks notebooks/jobs from ADF.
* Implement **dependency management, retries, timeout and failure handling**.
* Maintain pipeline metadata/control tables.

### 3. Bronze Layer – Raw Data

* Ingest data into **ADLS/Delta Bronze**.
* Keep data as close to source as possible.
* Add audit columns:

  * `ingestion_timestamp`
  * `source_file`
  * `batch_id`
* Handle schema evolution where required.

### 4. Databricks – Transformation

* Use **PySpark/SQL** for transformations.
* Clean nulls, duplicates and invalid records.
* Apply business rules.
* Perform joins, aggregations and enrichment.
* Optimize using:

  * Partition pruning
  * Broadcast joins
  * Appropriate partitioning
  * Caching only when useful
  * Delta optimization

### 5. Silver Layer

* Store **cleaned and standardized data** in Delta tables.
* Apply data quality checks.
* Implement CDC/SCD logic where required.
* Maintain a reliable, query-ready representation of business entities.

### 6. Gold Layer

* Create **business-ready aggregated tables**.
* Build fact/dimension models or analytical datasets.
* Optimize for BI/reporting workloads.
* Connect Power BI or downstream consumers.

### 7. Incremental Processing

* Avoid processing the entire dataset every time.
* Use:

  * Watermark columns
  * CDC
  * ADF Lookup/control tables
  * Delta Change Data Feed where appropriate
* Process only **new/changed records**.

### 8. Data Quality & Reconciliation

Validate:

* Source vs target record counts
* Duplicate records
* Null/mandatory fields
* Data types
* Business rules
* Control totals/checksums

Invalid records → **quarantine/error table**.

### 9. Error Handling & Monitoring

* ADF handles orchestration failures and retries.
* Databricks handles transformation errors.
* Maintain audit/error logs.
* Configure alerts through Azure monitoring/notifications.
* Track:
  **Pipeline → Batch → Notebook → Table → Status**

### 10. Security & Governance

* Azure Key Vault for secrets.
* Managed Identity/service principals.
* RBAC and Unity Catalog.
* Separate **DEV → UAT → PROD** environments.
* Implement data lineage and access controls.

### 11. Performance Optimization

For large datasets:

* Proper partition strategy
* Predicate/filter pushdown
* Partition pruning
* Broadcast small tables
* Reduce unnecessary shuffles
* Avoid excessive `repartition()`
* Use Delta optimization techniques
* Tune Spark cluster size/configuration.

### 12. CI/CD & Deployment

* Store ADF and Databricks code in Git.
* Use Azure DevOps/GitHub pipelines.
* Parameterize environment-specific configurations.
* Deploy automatically across **DEV → UAT → PROD**.

### Interview-ready architecture

**Source → ADF → ADLS Bronze → Databricks/PySpark → Delta Silver → Delta Gold → Power BI/Downstream**

The key point to emphasize is:

> **ADF orchestrates the pipeline; Databricks performs scalable data processing; ADLS/Delta provides the storage layer; and monitoring, security, data quality and CI/CD make the solution production-ready.**



### 3. Performance vs Cost vs Reliability 

In an interview, explain it as a **three-way trade-off**:

| Area                | How to optimize                                                                  | Trade-off                                  |
| ------------------- | -------------------------------------------------------------------------------- | ------------------------------------------ |
| ⚡ **Performance**   | More compute, parallelism, partitioning, caching, optimized joins                | Higher cost                                |
| 💰 **Cost**         | Autoscaling, job clusters, right-sizing, spot/low-cost compute where appropriate | May increase runtime or reduce reliability |
| 🛡️ **Reliability** | Retries, checkpoints, idempotency, monitoring, HA, data validation               | More infrastructure and operational cost   |

### 1. Performance

* Use **partition pruning** and predicate pushdown.
* Choose appropriate **partitioning**.
* Use **Broadcast Join** for small tables.
* Reduce unnecessary **shuffles**.
* Optimize Delta tables.
* Use appropriate cluster size and autoscaling.
* Parallelize independent ADF activities.

**Example:** If a Spark job takes 5 hours, increasing cluster size may reduce it to 1 hour—but compute cost increases.

### 2. Cost

* Use **job clusters** instead of always-running clusters.
* Enable **autoscaling** when workload varies.
* Right-size workers and avoid over-provisioning.
* Use incremental processing instead of full loads.
* Avoid unnecessary caching and repeated transformations.
* Schedule non-critical workloads during cheaper compute periods where applicable.

### 3. Reliability

* Configure **ADF retries and timeouts**.
* Make pipelines **idempotent** so reruns don't create duplicates.
* Use **checkpoints** for streaming workloads.
* Maintain audit/control tables.
* Implement data-quality and reconciliation checks.
* Use monitoring and alerting.
* Design proper failure/recovery paths.

### ⭐ Strong interview answer

> **“I don't optimize only for performance. I balance performance, cost, and reliability based on the business SLA. For critical real-time workloads, I prioritize performance and reliability even if the cost is higher. For non-critical batch workloads, I optimize for cost using autoscaling, incremental processing and right-sized clusters. For reliability, I use retries, idempotent processing, checkpoints, monitoring and data-quality validation.”**

**Simple rule:**
**SLA-critical → Performance + Reliability**
**Cost-sensitive batch → Cost + Acceptable Performance**
**Production pipelines → Reliability is non-negotiable**


## 5. Scalable Systems for TB-Level Data — Main Points

1. **Distributed Processing** – Use Databricks/Spark for parallel processing.
2. **Scalable Storage** – Use ADLS + Delta Lake.
3. **Partitioning** – Partition data based on frequently filtered columns.
4. **Incremental/CDC Processing** – Process only new or changed data.
5. **Optimize Joins** – Broadcast small tables and reduce shuffle.
6. **File Optimization** – Avoid small files; optimize Delta tables.
7. **Autoscaling** – Dynamically scale Databricks compute based on workload.
8. **Fault Tolerance** – Use retries, checkpoints and idempotent processing.
9. **Performance Optimization** – Partition pruning, predicate pushdown and caching where useful.
10. **Monitoring** – Track pipeline performance, failures, data quality and cost.
11. **Security & Governance** – RBAC, Unity Catalog, encryption and data lineage.
12. **Cost Optimization** – Right-size clusters and use job clusters.




###  ⭐ Interview Answer

> **“For TB-level data, I use Databricks/Spark for distributed processing with ADLS and Delta Lake for scalable storage. I use partitioning, incremental/CDC processing, partition pruning, optimized joins, and file optimization for performance. I use autoscaling and right-sized job clusters for cost control, while retries, checkpoints, idempotency, data-quality checks, and monitoring ensure reliability.”**
